# A Tokenizer You Can Rebuild

Train byte merges, encode with learned ranks, and decode without losing text.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** subword-tokenization.

## What You Will Be Able to Do

- Implement a byte-BPE lifecycle
- Separate learned merges from special tokens
- Detect a tokenizer/checkpoint mismatch

## Bytes are the reversible starting point

A Unicode character is not necessarily one byte. Encode each training document with UTF-8, producing integers from 0 to 255. The initial vocabulary maps each integer to its one-byte value. All text is representable before learning any merge; no unknown token is required. Decoding concatenates bytes, then decodes UTF-8. A generated incomplete byte sequence may not be valid text, so our decoder raises rather than silently replacing it.

BPE compresses frequent adjacent pairs. Count pairs separately inside each document, select the highest count, and replace non-overlapping occurrences left to right. Our deterministic tie rule chooses the lexicographically smallest pair. The new ID is 256 plus the merge index. Stop when there are no pairs or the requested merge count is reached. Never invent adjacency between two different documents.

## Training frequency is not the inference rule

Encoding new text replays the learned pair merges in their learned order. It does not count new frequencies and retrain. A pair formed by an earlier merge can participate in a later merge. The vocabulary and ordered merges are part of the model contract: logits at index 257 mean whatever this exact tokenizer assigned to 257.

This educational tokenizer has no regex pre-tokenization or special tokens. Production tokenizers often constrain merge boundaries and reserve control tokens. Untrusted text containing a special-token spelling must remain ordinary text unless explicitly admitted as a control token. A round-trip test alone cannot establish checkpoint compatibility; two tokenizers can both round-trip while assigning different IDs.

## Worked Example: Two merges on a tiny corpus

Train on ["abab", "ab"], requesting two merges.

1. **Count within documents:** (97,98) occurs three times; (98,97) once. The end of the first document is not joined to the start of the second.
2. **First merge:** Assign 256 to b"ab". The token lists become [256,256] and [256].
3. **Second merge:** Assign 257 to pair (256,256), meaning b"abab". Encoding "ababab" yields [257,256]. Concatenate their bytes to recover the original text.

Persist vocabulary and merge order with the checkpoint, not just vocabulary size.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Deterministic educational byte BPE. No regex splitting or special tokens."""
from collections import Counter

def _merge(ids, pair, new_id):
    result, i = ([], 0)
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result

def train_bpe(documents, num_merges):
    """Return (ordered merges, vocabulary).

documents: list[str]; num_merges: nonnegative integer.
merges: list[(left_id, right_id)], new ID = 256 + list index.
vocabulary: dict[int, bytes], including all 256 initial bytes.
Count pairs only within documents; highest count wins, lexicographic tie.
Replace non-overlapping pairs left to right. Stop when no pairs remain.
Raise ValueError for negative/noninteger num_merges. Do not mutate inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def encode_bpe(text, merges):
    """Encode a str using the saved ordered merges; never retrain on this text."""
    raise NotImplementedError('Implement this function, then run the checks.')

def decode_bpe(ids, vocabulary):
    """Concatenate vocabulary bytes and decode strict UTF-8 (no replacement)."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

The initial byte vocabulary has 256 entries. Twelve merges are successfully learned.

- Final vocabulary size

<details><summary>Worked answers</summary>

- 268: Each successful merge introduces exactly one ID: 256 + 12.

</details>

## Experiment

Run `experiments/scratch_tokenizer.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Compare token counts for training-like text and unfamiliar Unicode text. Compression is not model quality.
- Learn the same number of merges on a different corpus and inspect which IDs change. Store the exact artifact before any model training.

## Transfer Problems

1. Add reserved control-token IDs without allowing merges across control boundaries.
2. Write a compatibility check that rejects a same-size vocabulary with swapped IDs.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Karpathy: minbpe](https://github.com/karpathy/minbpe). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Deterministic educational byte BPE. No regex splitting or special tokens."""
from collections import Counter

def _merge(ids, pair, new_id):
    result, i = ([], 0)
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result

def train_bpe(documents, num_merges):
    """Return (ordered merges, vocabulary).

documents: list[str]; num_merges: nonnegative integer.
merges: list[(left_id, right_id)], new ID = 256 + list index.
vocabulary: dict[int, bytes], including all 256 initial bytes.
Count pairs only within documents; highest count wins, lexicographic tie.
Replace non-overlapping pairs left to right. Stop when no pairs remain.
Raise ValueError for negative/noninteger num_merges. Do not mutate inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def encode_bpe(text, merges):
    """Encode a str using the saved ordered merges; never retrain on this text."""
    raise NotImplementedError('Implement this function, then run the checks.')

def decode_bpe(ids, vocabulary):
    """Concatenate vocabulary bytes and decode strict UTF-8 (no replacement)."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
original = ['abab', 'ab']
merges, vocabulary = train_bpe(original, 2)
assert merges == [(97, 98), (256, 256)], 'Learn frequency-ranked pairs inside documents'
assert original == ['abab', 'ab'], 'Do not mutate training documents'
assert vocabulary[257] == b'abab' and len(vocabulary) == 258
assert encode_bpe('ababab', merges) == [257, 256], 'Replay learned ranks'
assert train_bpe(['a', 'b'], 4)[0] == [], 'Never merge across documents'
assert train_bpe(['ba', 'ab'], 1)[0] == [(97, 98)], 'Tie rule must be deterministic'
assert train_bpe(['aaaaa'], 1)[0] == [(97, 97)]
assert encode_bpe('aaaaa', [(97, 97)]) == [256, 256, 97], 'Non-overlapping replacements'
for sample in ['', 'new words', 'caf\u00e9', '\U0001f30d', 'abab', 'a\x00b']:
    assert decode_bpe(encode_bpe(sample, merges), vocabulary) == sample
for n in [-1, 1.5, True]:
    try:
        train_bpe(['abc'], n)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid merge counts')
try:
    decode_bpe([255], vocabulary)
except UnicodeDecodeError:
    pass
else:
    raise AssertionError('Do not silently replace malformed generated UTF-8')
print('Tokenizer lifecycle checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Deterministic educational byte BPE. No regex splitting or special tokens."""
from collections import Counter


def _merge(ids, pair, new_id):
    result, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result


def train_bpe(documents, num_merges):
    """Return (ordered merges, vocabulary).

    documents: list[str]; num_merges: nonnegative integer.
    merges: list[(left_id, right_id)], new ID = 256 + list index.
    vocabulary: dict[int, bytes], including all 256 initial bytes.
    Count pairs only within documents; highest count wins, lexicographic tie.
    Replace non-overlapping pairs left to right. Stop when no pairs remain.
    Raise ValueError for negative/noninteger num_merges. Do not mutate inputs.
    """
    if not isinstance(num_merges, int) or isinstance(num_merges, bool) or num_merges < 0:
        raise ValueError('num_merges must be a nonnegative integer')
    sequences = [list(text.encode('utf-8')) for text in documents]
    vocab = {i: bytes([i]) for i in range(256)}
    merges = []
    for _ in range(num_merges):
        counts = Counter(pair for ids in sequences for pair in zip(ids, ids[1:]))
        if not counts:
            break
        pair = min(counts, key=lambda p: (-counts[p], p))
        new_id = 256 + len(merges)
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        merges.append(pair)
        sequences = [_merge(ids, pair, new_id) for ids in sequences]
    return merges, vocab


def encode_bpe(text, merges):
    """Encode a str using the saved ordered merges; never retrain on this text."""
    ids = list(text.encode('utf-8'))
    for rank, pair in enumerate(merges):
        ids = _merge(ids, tuple(pair), 256 + rank)
    return ids


def decode_bpe(ids, vocabulary):
    """Concatenate vocabulary bytes and decode strict UTF-8 (no replacement)."""
    return b''.join(vocabulary[i] for i in ids).decode('utf-8')


In [ ]:
original = ['abab', 'ab']
merges, vocabulary = train_bpe(original, 2)
assert merges == [(97, 98), (256, 256)], 'Learn frequency-ranked pairs inside documents'
assert original == ['abab', 'ab'], 'Do not mutate training documents'
assert vocabulary[257] == b'abab' and len(vocabulary) == 258
assert encode_bpe('ababab', merges) == [257, 256], 'Replay learned ranks'
assert train_bpe(['a', 'b'], 4)[0] == [], 'Never merge across documents'
assert train_bpe(['ba', 'ab'], 1)[0] == [(97, 98)], 'Tie rule must be deterministic'
assert train_bpe(['aaaaa'], 1)[0] == [(97, 97)]
assert encode_bpe('aaaaa', [(97, 97)]) == [256, 256, 97], 'Non-overlapping replacements'
for sample in ['', 'new words', 'caf\u00e9', '\U0001f30d', 'abab', 'a\x00b']:
    assert decode_bpe(encode_bpe(sample, merges), vocabulary) == sample
for n in [-1, 1.5, True]:
    try:
        train_bpe(['abc'], n)
    except ValueError:
        pass
    else:
        raise AssertionError('Reject invalid merge counts')
try:
    decode_bpe([255], vocabulary)
except UnicodeDecodeError:
    pass
else:
    raise AssertionError('Do not silently replace malformed generated UTF-8')
print('Tokenizer lifecycle checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Original byte-BPE experiment; reference functions are embedded in the notebook."""
from pathlib import Path
import runpy

if 'train_bpe' not in globals():
    globals().update(runpy.run_path(str(Path(__file__).resolve().parents[1] / 'public/content/solutions/c10/01_tokenizer.py')))

corpus = ['a model learns a pattern', 'a pattern repeats', 'the model repeats a pattern']
for count in [0, 8, 24]:
    merges, vocab = train_bpe(corpus, count)
    for text in ['a model repeats a pattern', 'caf\u00e9 beyond the corpus']:
        ids = encode_bpe(text, merges)
        assert decode_bpe(ids, vocab) == text
        print({'merges': len(merges), 'bytes': len(text.encode('utf-8')), 'tokens': len(ids), 'text': text})
other, _ = train_bpe(['zzzzzyyyyyxxxx'], 8)
assert other != train_bpe(corpus, 8)[0]
print('Compression and round-trip correctness do not establish language-model quality.')
